# Lab 5: Campaign Agent with MCP Integration and AgentCore Deployment
## Build a campaign recommendation agent using MCP tools and deploy to production

## Overview

In this final lab, you'll build a sophisticated campaign recommendation agent that leverages the MCP tools from Lab 4 to generate targeted marketing campaigns. The agent will use the campaign analytics tools to retrieve player context, recommend items, and calculate pricing strategies.

### What You'll Build

**Campaign Recommendation Agent** with:

- **MCP Tools Integration**: Connect to Lab 4's campaign analytics tools via AgentCore Gateway
- **Advanced Prompt Engineering**: Specialized prompts optimized for the three MCP tools
- **Campaign Intelligence**: Generate targeted campaigns with specific items, pricing, and strategies
- **Production Deployment**: Host the agent on Amazon Bedrock AgentCore Runtime
- **End-to-End Workflow**: From player analysis to actionable campaign recommendations

## Prerequisites

To execute this lab you will need:
* **Lab 4 completed** - MCP Gateway with campaign analytics tools deployed
* AWS credentials configured
* Amazon Bedrock AgentCore access
* Access to Amazon Bedrock Claude models

> **How to run:** Click each code cell and press Shift+Enter or the Run button. Execute cells in order from top to bottom.

## Step 1: Install Dependencies

First, let's install the required packages for SageMaker environment.

In [ ]:
# Install required packages
!pip install strands-agents bedrock-agentcore-starter-toolkit "mcp[cli]" --q
print("✅ Dependencies installed successfully")

## Step 2: Environment Setup

Configure the development environment and verify AWS credentials for AgentCore deployment. This establishes the necessary connections and validates permissions required for deploying agents to the serverless runtime environment.

In [ ]:
import os, glob
# Remove stale AgentCore toolkit config so the campaign build can't accidentally pick up Lab 3's entrypoint.
for f in ['.bedrock_agentcore.yaml','Dockerfile','.dockerignore']:
    if os.path.exists(f):
        os.remove(f); print("removed", f)

In [ ]:
import sys
import os
import json
import boto3
from datetime import datetime

# Add utils to path
current_dir = os.getcwd()
utils_dir = os.path.join(current_dir, 'utils')
sys.path.insert(0, utils_dir)

import utils.gateway_utils as utils

# Configuration
REGION = os.environ.get('AWS_DEFAULT_REGION') or boto3.Session().region_name or 'us-west-2'
GATEWAY_URL = ""  # Will be loaded from SSM
ACCESS_TOKEN = ""  # Will be generated

print(f"Using AWS region: {REGION}")
print("✅ Environment setup complete")

## Step 3: Retrieve Gateway Configuration from Lab 4

Load the MCP gateway configuration saved in Lab 4 from AWS Systems Manager Parameter Store. This includes the gateway URL, Cognito authentication details, and OAuth credentials needed to access the campaign analytics tools.

In [ ]:
# Fetch gateway configuration from SSM Parameter Store
ssm_client = boto3.client('ssm', region_name=REGION)

try:
    response = ssm_client.get_parameter(
        Name='/game-analytics/gateway-config',
        WithDecryption=True
    )
    
    gateway_config = json.loads(response['Parameter']['Value'])
    
    # Extract only what we need for MCP connection and token generation
    GATEWAY_URL = gateway_config['gateway_url']
    USER_POOL_ID = gateway_config['user_pool_id']
    CLIENT_ID = gateway_config['client_id']
    CLIENT_SECRET = gateway_config['client_secret']
    SCOPE_STRING = gateway_config['scope_string']
    
    print(f"✅ Minimal gateway configuration loaded from SSM")
    print(f"Gateway URL: {GATEWAY_URL}")
    
except Exception as e:
    print(f"❌ Failed to load gateway config from SSM: {e}")
    print("Please run Lab 4 first to create the gateway configuration")

In [ ]:
# Get access token
if CLIENT_SECRET:
    token_response = utils.get_token(USER_POOL_ID, CLIENT_ID, CLIENT_SECRET, SCOPE_STRING, REGION)
    if 'access_token' in token_response:
        ACCESS_TOKEN = token_response["access_token"]
        print("✅ Token acquired successfully!")
    else:
        print(f"❌ Token acquisition failed: {token_response}")
else:
    print("⚠️ CLIENT_SECRET not available from gateway config.")

## Step 4: Create Campaign Agent with MCP Integration

Build the campaign recommendation agent with advanced prompt engineering optimized for the three MCP tools from Lab 4. The agent will use get_player_context_by_criteria for player analysis, recommend_items_for_segment for item discovery, and calculate_pricing_strategy for pricing optimization.

In [ ]:
import httpx2
from strands.models import BedrockModel
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_mcp_transport():
    http_client = httpx2.AsyncClient(
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
        timeout=httpx2.Timeout(30.0, read=300.0),
    )
    return streamable_http_client(GATEWAY_URL, http_client=http_client)

def create_campaign_agent_with_mcp(tools):
    """Campaign recommendation agent optimized for MCP tools from Lab 4"""
    model = BedrockModel(model_id="global.anthropic.claude-sonnet-5-5")
    
    system_prompt = """You are an expert Campaign Recommendation AI for a game. Your role is to analyze player segments and generate targeted marketing campaigns using the available MCP tools.

    AVAILABLE MCP TOOLS:
    1. **get_player_context_by_criteria**: Retrieves comprehensive player data using SQL criteria
    - Use SQL WHERE clauses to filter players (e.g., 'platform = "PC" AND churn_risk_score < 0.3')
    - Queryable columns: player_id, platform, faction_allegiance, geographic_region, churn_risk_score, engagement_intensity_score, total_spend_usd, total_transactions, avg_session_duration, session_count
    - platform values: 'PC', 'iOS', 'Android'
    - faction_allegiance values: 'Corporate Alliance', 'Neon Rebels', 'Void Hunters'
    - Returns player profiles across all datasets (profiles, spending, sessions, attributes)

    2. **recommend_items_for_segment**: Finds relevant items using semantic search
    - Use natural language queries (e.g., 'premium weapons for high spenders')
    - Specify campaign_goal: 'monetization', 'retention', or 'engagement'
    - Returns ranked items with metadata and relevance scores

    3. **calculate_pricing_strategy**: Optimizes pricing for recommended items
    - Input items array with base prices
    - Apply pricing instructions: discount, markup, or fixed_price
    - Returns final prices with transparency

    CAMPAIGN GENERATION WORKFLOW:
    1. **Analyze Target Segment**: Use get_player_context_by_criteria to understand player characteristics
    2. **Discover Relevant Items**: Use recommend_items_for_segment with segment-specific queries
    3. **Optimize Pricing**: Use calculate_pricing_strategy based on segment spending patterns
    4. **Generate Campaign**: Provide specific recommendations with rationale

    CAMPAIGN OUTPUT FORMAT:
    For each campaign, provide:
    - **Target Segment**: Clear description with player characteristics
    - **Recommended Items**: Specific items with names, categories, and relevance
    - **Pricing Strategy**: Final prices with discount/markup rationale
    - **Campaign Timing**: Optimal timing based on player behavior
    - **Expected Results**: Conversion rates and revenue projections
    - **Implementation**: Actionable steps for campaign execution

    Always use the MCP tools to gather data before making recommendations. Be specific and actionable."""
    
    return Agent(model=model, system_prompt=system_prompt, tools=tools)

print("✅ Campaign agent factory created")

## Step 5: Test Campaign Agent with MCP Tools

Test the campaign agent's ability to use the MCP tools for generating comprehensive campaign recommendations. This validates the agent can properly connect to the gateway, discover tools, and execute the complete workflow from player analysis to pricing optimization.

In [ ]:
# Test MCP connection and campaign agent
if ACCESS_TOKEN and GATEWAY_URL:
    client = MCPClient(create_mcp_transport)
    with client:
        tools = client.list_tools_sync()
        print(f"Available MCP tools: {[tool.tool_name for tool in tools]}")
        
        # Test campaign agent with comprehensive workflow
        print("\n=== Testing Campaign Agent with MCP Tools ===")
        campaign_agent = create_campaign_agent_with_mcp(tools)
        
        test_prompt = """Generate a targeted campaign for high-spending PC players. 
        Use the MCP tools to:
        1. Analyze high-value PC players (spending > $50)
        2. Find premium items suitable for this segment
        3. Calculate optimal pricing with appropriate discounts
        4. Provide a complete campaign recommendation"""
        
        campaign_response = campaign_agent(test_prompt)
        print(f"Campaign Agent Response: {str(campaign_response)}")
else:
    print("⚠️ Skipping tests - missing credentials")

## Step 6: Create Campaign Agent Deployment Files

Create the AgentCore-compatible deployment file for our campaign agent. This file includes the MCP client setup, tool integration, and the optimized system prompt for production deployment.

In [ ]:
%%writefile agentcore_app_campaign_agent.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import httpx2
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent
import boto3
import json
import sys
import os

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import utils.gateway_utils as utils

app = BedrockAgentCoreApp()
AWS_REGION = "us-west-2"

def get_gateway_config():
    ssm = boto3.client('ssm', region_name=AWS_REGION)
    response = ssm.get_parameter(Name='/game-analytics/gateway-config', WithDecryption=True)
    config = json.loads(response['Parameter']['Value'])
    
    # Generate fresh access token using utils
    token_response = utils.get_token(
        config['user_pool_id'],
        config['client_id'], 
        config['client_secret'],
        config['scope_string'],
        AWS_REGION
    )
    access_token = token_response['access_token']
    
    return config['gateway_url'], access_token

@app.entrypoint
def campaign_agent(payload):
    user_input = payload.get("prompt", "Generate campaign recommendations")
    
    if user_input.lower() == "ping":
        return {"status": "healthy"}
    
    gateway_url, access_token = get_gateway_config()
    
    def create_transport():
        http_client = httpx2.AsyncClient(
            headers={"Authorization": f"Bearer {access_token}"},
            timeout=httpx2.Timeout(30.0, read=300.0),
        )
        return streamable_http_client(gateway_url, http_client=http_client)
    
    client = MCPClient(create_transport)
    
    with client:
        tools = client.list_tools_sync()
        model = BedrockModel(model_id="global.anthropic.claude-sonnet-5-5")
        
        system_prompt = """You are an expert Campaign Recommendation AI for a game. Your role is to analyze player segments and generate targeted marketing campaigns using the available MCP tools.

        AVAILABLE MCP TOOLS:
        1. **get_player_context_by_criteria**: Retrieves comprehensive player data using SQL criteria
        - Use SQL WHERE clauses to filter players (e.g., 'platform = "PC" AND churn_risk_score < 0.3')
        - Queryable columns: player_id, platform, faction_allegiance, geographic_region, churn_risk_score, engagement_intensity_score, total_spend_usd, total_transactions, avg_session_duration, session_count
        - platform values: 'PC', 'iOS', 'Android'
        - faction_allegiance values: 'Corporate Alliance', 'Neon Rebels', 'Void Hunters'
        - Returns player profiles across all datasets (profiles, spending, sessions, attributes)

        2. **recommend_items_for_segment**: Finds relevant items using semantic search
        - Use natural language queries (e.g., 'premium weapons for high spenders')
        - Specify campaign_goal: 'monetization', 'retention', or 'engagement'
        - Returns ranked items with metadata and relevance scores

        3. **calculate_pricing_strategy**: Optimizes pricing for recommended items
        - Input items array with base prices
        - Apply pricing instructions: discount, markup, or fixed_price
        - Returns final prices with transparency

        CAMPAIGN GENERATION WORKFLOW:
        1. **Analyze Target Segment**: Use get_player_context_by_criteria to understand player characteristics
        2. **Discover Relevant Items**: Use recommend_items_for_segment with segment-specific queries
        3. **Optimize Pricing**: Use calculate_pricing_strategy based on segment spending patterns
        4. **Generate Campaign**: Provide specific recommendations with rationale

        CAMPAIGN OUTPUT FORMAT:
        For each campaign, provide:
        - **Target Segment**: Clear description with player characteristics
        - **Recommended Items**: Specific items with names, categories, and relevance
        - **Pricing Strategy**: Final prices with discount/markup rationale
        - **Campaign Timing**: Optimal timing based on player behavior
        - **Expected Results**: Conversion rates and revenue projections
        - **Implementation**: Actionable steps for campaign execution

        Always use the MCP tools to gather data before making recommendations. Be specific and actionable."""

        agent = Agent(model=model, system_prompt=system_prompt, tools=tools)
        
        response = agent(user_input)
        return {"result": response.message}

if __name__ == "__main__":
    app.run()

### Create Requirements File

Define the Python dependencies needed for our agent to run in the AgentCore Runtime environment. This includes the Strands framework for agent development, MCP for protocol support, and AWS SDKs for vector operations and player analytics.

In [ ]:
%%writefile requirements.txt
strands-agents>=1.65.0
bedrock-agentcore
bedrock-agentcore-starter-toolkit
mcp[cli]==2.3.0
httpx2
boto3>=1.26.0
pandas>=1.3.0
matplotlib>=3.4.0

## Step 7: Deploy Campaign Agent to AgentCore Runtime

Deploy the campaign agent to Amazon Bedrock AgentCore Runtime with the MCP gateway configuration as environment variables. This creates a production-ready endpoint for generating campaign recommendations using the MCP tools.

In [ ]:
import boto3
import json

def create_agentcore_role(agent_name):
    """Create IAM role for AgentCore Runtime with S3 Vectors permissions."""
    iam = boto3.client('iam')
    
    role_name = f"{agent_name}-agentcore-role"
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock-agentcore.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"IAM role for {agent_name} AgentCore Runtime"
        )
        
        # Attach Bedrock permissions
        iam.attach_role_policy(
            RoleName=role_name,
            PolicyArn="arn:aws:iam::aws:policy/AmazonBedrockFullAccess"
        )
        
        # Add SSM permissions for gateway config
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="SSMParameterAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": [
                        "ssm:GetParameter",
                        "ssm:GetParameters"
                    ],
                    "Resource": "arn:aws:ssm:*:*:parameter/game-analytics/*"
                }]
            })
        )
        
        # Add Cognito permissions for token generation
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="CognitoAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": [
                        "cognito-idp:InitiateAuth"
                    ],
                    "Resource": "*"
                }]
            })
        )
        
        # Add ECR permissions
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName="ECRAccess",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow", 
                    "Action": [
                        "ecr:GetAuthorizationToken",
                        "ecr:BatchGetImage", 
                        "ecr:GetDownloadUrlForLayer"
                    ], 
                    "Resource": "*"
                }]
            })
        )
        
        print(f"✅ Created IAM role: {role_name}")
        return role
        
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=role_name)
        print(f"✅ Using existing IAM role: {role_name}")
        return role

agent_name = "campaign_agent"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

### Configure and Launch Agent

Configure the AgentCore Runtime with our agent code, requirements, and IAM role, then launch it to create a production-ready endpoint. The runtime handles containerization, deployment, and scaling automatically.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="agentcore_app_campaign_agent.py",
    auto_create_execution_role=False,
    execution_role=agentcore_iam_role['Role']['Arn'],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name
)

print("✅ AgentCore Runtime configured")
print(f"Configuration: {response}")

print("🚀 Launching agent to AgentCore Runtime...")
campaign_launch_result = agentcore_runtime.launch()

print("✅ Agent launched successfully!")
print(f"Agent ARN: {campaign_launch_result.agent_arn}")
print(f"ECR URI: {campaign_launch_result.ecr_uri}")
CAMPAIGN_AGENT_ARN = campaign_launch_result.agent_arn

## Step 8: Monitor Deployment Status

Monitor the campaign agent deployment progress and wait for it to become ready for invocation. This ensures the agent is fully deployed and accessible before testing.

In [ ]:
import time

print("⏳ Monitoring campaign agent deployment status...")

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    print(f"Status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

print(f"✅ Final Status: {status}")

if status == 'READY':
    print("🎉 Campaign Agent is ready for invocation!")
else:
    print(f"❌ Deployment failed with status: {status}")

## Step 9: Test Deployed Campaign Agent

Test the deployed campaign agent with a comprehensive campaign generation request that exercises all three MCP tools. This validates the complete workflow from player analysis to campaign recommendations in the production environment.

In [ ]:
import uuid

if status == 'READY':
    agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
    
    # Test Campaign Agent with comprehensive workflow
    print("🧪 Testing Deployed Campaign Agent...")
    campaign_session_id = f"campaign-session-{uuid.uuid4().hex}"
    
    comprehensive_test_prompt = """Generate a targeted campaign for the following player segment:

    Segment: Neon Rebels Mobile Spenders
    Description: Mobile platform Neon Rebels players who show strong spending behavior and faction loyalty
    Criteria: SELECT player_id FROM player_profiles p JOIN lifetime_spend s ON p.player_id = s.player_id WHERE p.faction_allegiance = 'Neon Rebels' AND p.platform IN ('iOS', 'Android') AND s.total_spend_usd > 10
    """
    
    campaign_test_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=CAMPAIGN_AGENT_ARN,
        runtimeSessionId=campaign_session_id,
        payload=json.dumps({
            "prompt": comprehensive_test_prompt
        }),
        qualifier="DEFAULT"
    )
    
    campaign_response_body = campaign_test_response['response'].read().decode('utf-8')
    campaign_response_data = json.loads(campaign_response_body)
    blocks = campaign_response_data['result']['content']
    agent_text = next(b['text'] for b in blocks if 'text' in b)
    
    print("\n=== Campaign Agent Response ===")
    print(agent_text)
    
else:
    print("⚠️ Cannot test agent - deployment not ready")

## Step 10: Display Agent ARN

Display the deployed agent ARN for use in applications.

In [ ]:
print(f"Campaign Agent ARN: {CAMPAIGN_AGENT_ARN}")

## Congratulations! 🎉

You have successfully:

✅ **Built campaign agent** with advanced MCP integration and optimized prompts  
✅ **Integrated MCP tools** for player analysis, item recommendations, and pricing optimization  
✅ **Deployed to production** on Amazon Bedrock AgentCore Runtime with environment configuration  
✅ **Tested complete workflow** from player context to campaign recommendations  
✅ **Created production endpoint** ready for application integration

### Key Concepts Learned:
- **MCP Integration**: Using Model Context Protocol tools in production agents
- **Advanced Prompt Engineering**: Optimizing prompts for specific tool workflows
- **Production Deployment**: Environment variables and configuration management
- **End-to-End Campaigns**: Complete workflow from analysis to recommendations

---

## 🎯 Next Steps: Test Your Agent in the Frontend

Now that your campaign agent is deployed with MCP tools, you can test it using the Streamlit frontend:

1. **Copy the Campaign Agent ARN** from the deployment output above (it looks like: `arn:aws:bedrock-agentcore:us-west-2:123456789012:agent/xyz789...`)

2. **Open the Streamlit frontend** and navigate to the **Campaign Recommender** tab

3. **Paste the Campaign Agent ARN** into the agent configuration section

4. **Try asking the campaign agent** to generate targeted campaigns.

5. **Watch the agent use MCP tools** to create personalized campaigns with carefully chosen items and optimized pricing